In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
raw_csv_path = 'data/raw/cybersecurity.csv'
df_cleaned = pd.read_csv(raw_csv_path)
train_df, test_df = train_test_split(df_cleaned, test_size=0.3, random_state=42)

In [3]:
def print_df_info(df):
    print("=" * 50)
    print(f"Shape: {df.shape}")
    print("=" * 50)
    print("Columns:")
    print(df.columns.tolist())
    print("=" * 50)
    print("Statistical description:")
    print(df.describe())
    print("=" * 50)
    print("First 5 rows:")
    print(df.head(5))
    print("=" * 50)
    print("Last 5 rows:")
    print(df.tail(5))
    print("=" * 50)
    print("amount of unique values:")
    print(df.nunique())
    print("=" * 50)

In [4]:
print_df_info(train_df)

Shape: (7000, 13)
Columns:
['timestamp', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'bytes_sent', 'bytes_received', 'user_agent', 'url', 'is_internal_traffic', 'label', 'attack_type']
Statistical description:
           src_port      dst_port    bytes_sent  bytes_received        label
count   7000.000000   7000.000000  7.000000e+03    7.000000e+03  7000.000000
mean   27294.993857   2006.131714  1.367411e+05    2.810714e+05     0.039714
std    20990.842941   6658.759360  2.155574e+06    4.329112e+06     0.195301
min       21.000000      5.000000  1.700000e+01    3.000000e+00     0.000000
25%     6391.250000     25.000000  7.123750e+03    1.097600e+04     0.000000
50%    25870.000000    443.000000  2.130750e+04    3.683500e+04     0.000000
75%    45526.000000   1433.000000  6.409300e+04    1.206655e+05     0.000000
max    65524.000000  65491.000000  1.391704e+08    2.914830e+08     1.000000
First 5 rows:
                timestamp           src_ip           dst_ip  src_port  

In [5]:
print_df_info(test_df)

Shape: (3000, 13)
Columns:
['timestamp', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'bytes_sent', 'bytes_received', 'user_agent', 'url', 'is_internal_traffic', 'label', 'attack_type']
Statistical description:
           src_port      dst_port    bytes_sent  bytes_received        label
count   3000.000000   3000.000000  3.000000e+03    3.000000e+03  3000.000000
mean   26485.760000   2076.627667  1.517948e+05    2.177942e+05     0.040667
std    20943.170858   7020.840226  2.063022e+06    1.894420e+06     0.197550
min       21.000000     21.000000  8.200000e+01    3.000000e+01     0.000000
25%     5617.500000     25.000000  7.177500e+03    1.095025e+04     0.000000
50%    24366.000000    443.000000  2.266800e+04    3.920900e+04     0.000000
75%    44720.250000   1433.000000  6.866550e+04    1.224278e+05     0.000000
max    65429.000000  65416.000000  6.841120e+07    9.271670e+07     1.000000
First 5 rows:
                timestamp          src_ip          dst_ip  src_port  ds

In [6]:
def analyze_missing(df):
    missing = df.isnull().sum()
    missing_percent = 100 * missing / len(df)
    missing_df = pd.DataFrame({
        'Column': df.columns,
        'Missing' : missing.values,
        'Percent' : missing_percent.values,
        'Dtype': df.dtypes.values
    })
    missing_df = missing_df[missing_df['Missing'] > 0]
    if len(missing_df) == 0:
        print("No missing values in dataset")
    else:
        print(missing_df.to_string(index=False))

In [7]:
analyze_missing(train_df)

Column  Missing   Percent Dtype
   url     2249 32.128571   str


In [8]:
analyze_missing(test_df)

Column  Missing   Percent Dtype
   url      983 32.766667   str


In [9]:
def clean_df(df):
    print("Cleaning dataset")
    print("="*50)

    #Removing duplicates
    if df.duplicated().sum() > 0:
        dup_count = df.duplicated().sum()
        df = df.drop_duplicates()
        print(f"Removed {dup_count} duplicate rows \n")

    #Handle missing values
    if df.isnull().sum().sum() > 0:
        print("Handling missing values")
        print("=" * 50)

        numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
        for col in numeric_cols:
            if df[col].isnull().any():
                fill_value = df[col].median()
                df[col] = df[col].fillna(fill_value)
                print(f"{col}: Filled with median={fill_value}")

        if df['url'].isnull().any():
            df['url'] = df['url'].fillna('')
            print("url: Filled with empty string")

        if df['user_agent'].isnull().any():
            df['user_agent'] = df['user_agent'].fillna('Unknown')
            print("user_agent: Filled with 'Unknown'")

        cat_cols = ['src_ip', 'dst_ip', 'protocol', 'attack_type']
        for col in cat_cols:
            if col in df.columns and df[col].isnull().any():
                mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
                df[col] = df[col].fillna(mode_val)
                print(f"{col}: Filled with mode='{mode_val}'")
    else:
        print("No missing values detected - data is already clean!\n")

    final_missing = df.isnull().sum().sum()
    print(f"Missing values: {final_missing}")

In [10]:
clean_df(train_df)
clean_df(test_df)

Cleaning dataset
Handling missing values
url: Filled with empty string
Missing values: 0
Cleaning dataset
Handling missing values
url: Filled with empty string
Missing values: 0


In [ ]:
def data_normalization(train_df, test_df):
    print("\n" + "=" * 50)
    print("Data normalization")
    print("=" * 50)

    train = train_df.copy()
    test = test_df.copy()

    print("\n Processing timestamp")
    for df in [train, test]:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df["hour"] = df["timestamp"].dt.hour
        df["day"] = df["timestamp"].dt.day
        df["month"] = df["timestamp"].dt.month
        df["dayofweek"] = df["timestamp"].dt.dayofweek

    print("\n Normalizing bytes (StandardScaler)")
    scaler_bytes = StandardScaler()

    train[["bytes_sent", "bytes_received"]] = scaler_bytes.fit_transform(train[["bytes_sent", "bytes_received"]])

    test[["bytes_sent", "bytes_received"]] = scaler_bytes.transform(test[["bytes_sent", "bytes_received"]])

    print(f"Train mean: {train[['bytes_sent', 'bytes_received']].mean().values}")
    print(f"Test mean: {test[['bytes_sent', 'bytes_received']].mean().values}")

    print("\n Normalizing ports and time features (MinMaxScaler)")
    features_to_scale = ["src_port", "dst_port", "hour", "day", "month", "dayofweek"]

    scaler_ports = MinMaxScaler()

    train[features_to_scale] = scaler_ports.fit_transform(train[features_to_scale])

    test[features_to_scale] = scaler_ports.transform(test[features_to_scale])

    print("\n" + "=" * 50)
    print("Normalization completed!")
    print("=" * 50)

    return train, test

In [12]:
train_df, test_df = data_normalization(train_df, test_df)


Data normalization

 Processing timestamp

 Normalizing bytes (StandardScaler)
Train mean: [-3.04518315e-18 -5.07530526e-18]
Test mean: [ 0.00698409 -0.01461773]

 Normalizing ports and time features (MinMaxScaler)

Normalization completed!


In [17]:
def encoding(train_df, test_df):
    print("\n" + "=" * 50)
    print("Data encoding")
    print("=" * 50)

    train = train_df.copy()
    test = test_df.copy()
    print("\n Protocol encoding:")
   
    protocol_dummies_train = pd.get_dummies(train['protocol'], prefix = 'protocol')
    protocol_dummies_test = pd.get_dummies(test['protocol'], prefix = 'protocol')
    
    train = pd.concat([train, protocol_dummies_train], axis=1)
    test = pd.concat([test, protocol_dummies_test], axis=1)
    train.drop('protocol', axis=1, inplace=True)
    test.drop('protocol', axis=1, inplace=True)

    print(f"Train columns: {list(protocol_dummies_train.columns)}")
    print(f"Test columns: {list(protocol_dummies_test.columns)}")
    
    print("\n Attack type encoding:")
    le_attack = LabelEncoder()
    train['attack_type'] = le_attack.fit_transform(train['attack_type'])
    test['attack_type'] = le_attack.transform(test['attack_type'])
    print(f"Mapping: {dict(zip(le_attack.classes_, le_attack.transform(le_attack.classes_)))}")

    if 'url' in train.columns:
        print("\n URL encoding:")
        all_urls = pd.concat([train['url'], test['url']]).unique()
        le_url = LabelEncoder()
        le_url.fit(all_urls)
        
        train['url'] = train['url'].fillna('unknown')
        test['url'] = test['url'].fillna('unknown')
        
        train['url_encoded'] = le_url.transform(train['url'])
        test['url_encoded'] = le_url.transform(test['url'])
        
        train.drop('url', axis=1, inplace=True)
        test.drop('url', axis=1, inplace=True)
        
        print(f" Unique URLs in train: {train['url_encoded'].nunique()}")
        print(f" Unique URLs in test: {test['url_encoded'].nunique()}")

    if 'user_agent' in train.columns:
        print("\n User agent encoding:")
        all_agents = pd.concat([train['user_agent'], test['user_agent']]).unique()
        le_ua = LabelEncoder()
        le_ua.fit(all_agents)
        
        train['user_agent'] = train['user_agent'].fillna('unknown')
        test['user_agent'] = test['user_agent'].fillna('unknown')
        
        train['user_agent_encoded'] = le_ua.transform(train['user_agent'])
        test['user_agent_encoded'] = le_ua.transform(test['user_agent'])
        
        train.drop('user_agent', axis=1, inplace=True)
        test.drop('user_agent', axis=1, inplace=True)
        
        print(f"Unique agents in train: {train['user_agent_encoded'].nunique()}")
        print(f"Unique agents in test: {test['user_agent_encoded'].nunique()}")

    if 'src_ip' in train.columns:
        print("\n IP addresses encoding:")
        train['src_ip_first'] = train['src_ip'].astype(str).str.split('.').str[0].astype(int)
        train['dst_ip_first'] = train['dst_ip'].astype(str).str.split('.').str[0].astype(int)
        
        test['src_ip_first'] = test['src_ip'].astype(str).str.split('.').str[0].astype(int)
        test['dst_ip_first'] = test['dst_ip'].astype(str).str.split('.').str[0].astype(int)
        
        train.drop(['src_ip', 'dst_ip'], axis=1, inplace=True)
        test.drop(['src_ip', 'dst_ip'], axis=1, inplace=True)
        print("Extracted first octet from IPs")

    if 'is_internal_traffic' in train.columns:
        train['is_internal_traffic'] = train['is_internal_traffic'].astype(int)
        test['is_internal_traffic'] = test['is_internal_traffic'].astype(int)
    
    print("\n" + "=" * 50)
    print("Encoding completed!")
    print("=" * 50)
    print(f"\nTrain shape: {train.shape}")
    print(f"Test shape: {test.shape}")
    print(f"\nTrain columns: {list(train.columns)}")
    print(f"Test columns: {list(test.columns)}")
    return train, test


In [18]:
train_df, test_df = encoding(train_df, test_df)


Data encoding

 Protocol encoding:
Train columns: ['protocol_ICMP', 'protocol_TCP', 'protocol_UDP']
Test columns: ['protocol_ICMP', 'protocol_TCP', 'protocol_UDP']

 Attack type encoding:
Mapping: {'benign': np.int64(0), 'brute-force': np.int64(1), 'c2': np.int64(2), 'command-injection': np.int64(3), 'credential-stuffing': np.int64(4), 'ddos': np.int64(5), 'exploit-attempt': np.int64(6), 'port-scan': np.int64(7), 'sql-injection': np.int64(8), 'xss': np.int64(9)}

 URL encoding:
 Unique URLs in train: 4752
 Unique URLs in test: 2018

 User agent encoding:
Unique agents in train: 13
Unique agents in test: 13

 IP addresses encoding:
Extracted first octet from IPs

Encoding completed!

Train shape: (7000, 19)
Test shape: (3000, 19)

Train columns: ['timestamp', 'src_port', 'dst_port', 'bytes_sent', 'bytes_received', 'is_internal_traffic', 'label', 'attack_type', 'hour', 'day', 'month', 'dayofweek', 'protocol_ICMP', 'protocol_TCP', 'protocol_UDP', 'url_encoded', 'user_agent_encoded', 'src

In [23]:
train_df.to_csv("data/processed/train_processed.csv")
test_df.to_csv("data/processed/test_processed.csv")
print("Files saved")

Files saved
